# 09 — A controlled feature study

**Question.** Can broad feature engineering improve a compact ranker when the candidate pool,
historical cutoff, fitting sample, and evaluation cohort are held fixed?

**Result.** The selected 102-feature model achieves **0.58439 weighted Recall@20** on
**432,492 temporal evaluation sessions**. The compact 28-feature model scores **0.56490**;
fixed candidate fusion scores **0.53524**. Selection preferred removing source-score features.
Orders drive the gain; fusion still wins click and cart recall.

This notebook regenerates every chart from checksum-verified, committed evidence. Model fitting
and the independent event audit ran separately at full scale. [Notebook 10](10_competition_inference.ipynb)
runs native models and generates predictions. [Methods](../docs/RESEARCH.md) ·
[Machine-readable catalog](../reports/research/feature_catalog.csv) ·
[Independent audit](../reports/research/audit.json)

In [ ]:
from pathlib import Path
import hashlib
import json
import math
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'reports/research/manifest.json').is_file())
DATA = ROOT / 'reports/research'
load = lambda name: json.loads((DATA / name).read_text())
manifest = load('manifest.json')
for name, expected in manifest['files'].items():
    assert hashlib.sha256((DATA / name).read_bytes()).hexdigest() == expected, name
evaluation, ablations = load('evaluation.json'), load('ablations.json')
audit, interpretation = load('audit.json'), load('interpretation.json')
screening, seal = load('screening.json'), load('evaluation_seal.json')
assert len({x['seal_id'] for x in (manifest, evaluation, audit, interpretation, seal)}) == 1
assert audit['status'] == evaluation['status'] == interpretation['status'] == 'passed'
assert seal['evaluation_labels_consulted'] is False
catalog = pd.read_csv(DATA / 'feature_catalog.csv')
families = pd.read_csv(DATA / 'feature_families.csv')
models = pd.read_csv(DATA / 'ablation_models.csv')
importance = pd.read_csv(DATA / 'feature_importance.csv')
COLORS = {'selected': '#0F766E', 'core': '#2563EB', 'fusion': '#64748B', 'ceiling': '#D97706'}
pio.renderers.default = 'plotly_mimetype+png'

def show(fig, title, height=480, left=90, **layout):
    fig.update_layout(template='plotly_white', width=1000, height=height,
        title=dict(text=title, x=0.03), font=dict(family='Arial', size=14, color='#1E293B'),
        margin=dict(l=left, r=40, t=85, b=70), paper_bgcolor='white', plot_bgcolor='white',
        legend=dict(orientation='h', y=1.12, x=0), hoverlabel=dict(font_size=14))
    fig.update_layout(**layout)
    fig.show()

print(f"Verified {len(manifest['files'])} evidence files; evaluation seal {seal['seal_id'][:12]}.")

## Freeze time before choosing models

Every learned retrieval component in this study is newly fitted on events before **20 August 2022,
22:00 UTC**. Whole sessions are assigned by their first event, clipped at exclusive period ends,
and cut into observed prefixes and future targets without using labels for inclusion. Fit and
selection session samples use deterministic hashes; evaluation includes every eligible session.

The original OTTO data had earlier exploratory exposure. This is a **newly reserved temporal
evaluation**, not a claim that the underlying data had never been seen. The earlier neural and
Item2Vec artifacts are not reused in this controlled comparison.

In [ ]:
protocol = load('temporal_contract.json')['protocol']
periods = pd.DataFrame([
    ['Historical retrieval', 'before 2022-08-20 22:00 UTC', 'all permitted history'],
    ['Ranker fit', '2022-08-20 → 2022-08-23, 22:00 UTC', f"{audit['source']['roles']['fit']:,} sessions"],
    ['Model selection', '2022-08-23 → 2022-08-24, 22:00 UTC', f"{audit['source']['roles']['selection']:,} sessions"],
    ['Final evaluation', '2022-08-24 → 2022-08-26, 22:00 UTC', f"{evaluation['sessions']:,} sessions"],
], columns=['Role', 'Time boundary', 'Size'])
display(periods.style.hide(axis='index'))
assert all(v == 0 for v in audit['source']['differences'].values())
assert audit['native_replay']['mismatches'] == 0
print(f"Independent audit: {audit['source']['source_partitions']} original Parquet partitions; "
      f"{audit['source']['reconstructed_labels']:,} reconstructed targets; "
      f"{audit['native_replay']['comparisons']:,} sampled native-model/candidate checks; zero differences.")

## Engineer broadly; retain evidence of value

The catalog spans historical popularity/trends, observed recurrence and intent, source agreement,
session context, and action/time-weighted graph affinity. Its 1,482 entries are explicit formulas,
not identifiers or future-label features. A broad catalog is a search space, not a result by itself.

Screening uses **514,013 candidate rows from 8,448 fitting sessions**. Constant, near-constant, and
duplicate columns are removed first. Nine binary LightGBM pilots—three objectives across three
session-grouped folds—supply normalized gain and stability diagnostics. Protected compact features,
redundancy filtering, and a 128-column capacity limit define the shortlist. Binary log loss and
feature/target correlations are screening diagnostics, not Recall@20 estimates.

In [ ]:
eligible = len(catalog) - sum(screening['rejections'][k] for k in ('constant', 'near_constant', 'duplicate'))
fig = go.Figure(go.Funnel(y=['Engineered formulas', 'Pass quality checks', 'Fit-only shortlist', 'Selection-chosen model'],
    x=[len(catalog), eligible, screening['retained_count'], int(catalog['final_selected'].sum())],
    textinfo='value+percent initial', marker=dict(color=['#CBD5E1', '#94A3B8', '#2563EB', '#0F766E'])))
show(fig, '1,482 engineered features → 102 in the final model', 430)
display(families.rename(columns={'family':'Family', 'engineered':'Engineered', 'screened':'Shortlist', 'final':'Final'})
        .style.hide(axis='index'))
rejected = pd.DataFrame([{'Reason': k, 'Features': v} for k, v in screening['rejections'].items() if k != 'retained'])
display(rejected.style.hide(axis='index'))
retained = catalog[catalog['status'] == 'retained']
display(retained.groupby('family', as_index=False).agg(
    median_fold_stability=('positive_gain_fold_fraction', 'median'),
    minimum_fold_stability=('positive_gain_fold_fraction', 'min')).style.format(precision=3).hide(axis='index'))

## Test the shortlist under a matched ranking experiment

Eight variants × three objectives = **24 LambdaRank fits**. Every variant uses the same
100,000 fitting sessions, 400-candidate pools, and retained positives plus the same 30 hard/30
random fitting negatives. All 20,000 selection queries keep the full candidate pool and complete
target denominators. Selection maximizes official task Recall@20; fewer features, then name,
break ties. Checkpoints are measured at iteration 1 and every 5 rounds, with early stopping.

Removing the 26 shortlisted source-score features wins all three objectives in selection.
Graph and intent interactions still encode retrieval information; this ablation isolates the
direct source family. This is a useful negative result: screening importance did not guarantee ranking utility.
The other five families remain, giving **102 features**. Family comparisons are exploratory
model-selection evidence; their maximum is not an unbiased performance estimate.

In [ ]:
selection = pd.DataFrame({name: {**{o: row['objectives'][o]['recall_at_20'] for o in ('clicks','carts','orders')},
    'weighted': row['weighted_recall_at_20']} for name,row in ablations['variants'].items()}).T
selection = selection.sort_values('weighted', ascending=False)
fig = px.imshow(selection, text_auto='.4f', aspect='auto', color_continuous_scale='Blues',
    labels=dict(x='Objective', y='Variant', color='Recall@20'))
fig.update_xaxes(side='top', title_text=None)
show(fig, 'Selection scores: same 20,000 sessions and 400 candidates', 570)
display(models[models['chosen']][['objective','variant','features','best_iteration','fit_seconds']]
    .sort_values('objective').style.format({'fit_seconds':'{:.2f}'}).hide(axis='index'))
assert set(ablations['chosen'].values()) == {'without_source'}
assert all(seal['models'][o]['variant'] == ablations['chosen'][o] for o in ablations['chosen'])
print('All model choices were sealed before evaluation labels were opened.')

## Evaluate once, with every eligible query in the denominator

The competition metric pools hits and unique-target denominators capped at 20 within each
objective, then applies weights **0.10 / 0.30 / 0.60**. Unretrievable targets remain in the
denominator. Click targets are the next click; carts and orders are unique future items.

The selected model improves the compact ranker by **1.949 percentage points** and fusion by
**4.915 points**. The paired 95% bootstrap intervals exclude zero. These intervals resample
sessions within this frozen model/cohort; they do not include training-seed uncertainty or
uncertainty from the search over model variants.

In [ ]:
scores = pd.DataFrame({name: {**{o: row['objectives'][o]['recall_at_20'] for o in ('clicks','carts','orders')},
    'weighted': row['weighted_recall_at_20']} for name,row in evaluation['scores'].items()})
for name,row in evaluation['scores'].items():
    recomputed = sum(w * row['objectives'][o]['hits'] / row['objectives'][o]['denominator']
        for o,w in {'clicks':.1,'carts':.3,'orders':.6}.items())
    assert math.isclose(recomputed, row['weighted_recall_at_20'], abs_tol=1e-12)
fig = go.Figure()
for name in ('fusion','core','selected'):
    fig.add_bar(name=name.capitalize(), x=scores.index, y=scores[name], marker_color=COLORS[name],
        text=scores[name].map(lambda x: f'{x:.4f}'), textposition='outside')
fig.update_layout(barmode='group')
fig.update_yaxes(title='Recall@20', range=[0,.78])
show(fig, f"Reserved temporal evaluation · {evaluation['sessions']:,} sessions", 510)
gain_rows = []
for baseline, result in evaluation['paired_intervals'].items():
    gain_rows.append({'Comparison': f'Selected − {baseline}', 'Gain (pp)': 100*result['absolute_gain'],
        '95% lower (pp)':100*result['gain_interval'][0], '95% upper (pp)':100*result['gain_interval'][1]})
display(pd.DataFrame(gain_rows).style.format(precision=3).hide(axis='index'))
print('1,000 paired session bootstrap replicates; seed 20260908.')

**What did not improve?** Fusion still beats the selected model on clicks (0.52678 vs 0.50565)
and carts (0.43021 vs 0.42792). Orders improve from 0.58917 to 0.67575 and carry 60% of the official
metric. No post-evaluation hybrid was selected to hide those losses. The old Fold 0 score in
Notebook 08 uses a different cohort and pool; it is not a matched baseline for this study.

## Separate candidate coverage from ranking quality

The same nested 100/200/400-item pools have weighted candidate ceilings of 0.65603 / 0.67497 /
0.68695. A ceiling counts recoverable targets with an ideal ordering, capped at 20; it is not an
achieved prediction score. All learned comparisons above use 400 candidates. These curves do not
claim that a ranker was refitted at each smaller budget or that larger pools have free compute cost.

In [ ]:
budgets = [100,200,400]
fig = go.Figure()
for objective in ('clicks','carts','orders'):
    y = [evaluation['candidate_frontier'][str(k)]['objectives'][objective]['recall_at_20'] for k in budgets]
    fig.add_scatter(name=objective.capitalize()+' ceiling', x=budgets, y=y, mode='lines+markers')
fig.add_scatter(name='Weighted ceiling', x=budgets,
    y=[evaluation['candidate_frontier'][str(k)]['weighted_recall_at_20'] for k in budgets],
    mode='lines+markers', line=dict(color=COLORS['ceiling'], width=4))
fig.add_scatter(name='Selected ranked score at 400', x=[400],
    y=[evaluation['scores']['selected']['weighted_recall_at_20']], mode='markers',
    marker=dict(color=COLORS['selected'], size=16, symbol='diamond'))
fig.update_xaxes(title='Candidates per query', tickvals=budgets)
fig.update_yaxes(title='Recall / candidate ceiling', range=[.48,.80])
show(fig, 'Candidate coverage places an upper bound on ranking quality', 510)

## Explain the frozen model without reopening selection

Native LightGBM TreeSHAP explains raw ranking scores on **4,096 deterministic selection
candidate rows**. Additivity is checked against native model outputs; the largest absolute error
is below 6 × 10⁻¹⁵. These contributions are not calibrated probabilities or causal effects.
Each panel uses its own objective's score scale.

In [ ]:
fig = make_subplots(rows=3, cols=1, subplot_titles=['Clicks','Carts','Orders'], vertical_spacing=.10)
for row, objective in enumerate(('clicks','carts','orders'), 1):
    top = importance[importance['objective'] == objective].nlargest(8, 'mean_absolute_shap').sort_values('mean_absolute_shap')
    fig.add_bar(x=top['mean_absolute_shap'], y=top['feature'], orientation='h', row=row, col=1,
        marker_color=COLORS['selected'], showlegend=False,
        hovertemplate='%{y}<br>Mean |SHAP|: %{x:.5f}<extra></extra>')
    fig.update_yaxes(tickfont=dict(size=12), row=row, col=1)
    fig.update_xaxes(title='Mean |SHAP|', row=row, col=1)
show(fig, 'Most influential features in each task-specific model', 1100, left=300)
display(pd.DataFrame([{'Objective': o, 'Maximum additivity error': e}
    for o,e in interpretation['shap_additivity_max_abs_error'].items()])
    .style.format({'Maximum additivity error':'{:.2e}'}).hide(axis='index'))

Whole-query block permutation complements feature-level SHAP. Each family is shuffled twice
across **1,000 selection sessions**, preserving relationships within the family. Candidate/feature
dependencies can still be broken, so the measured recall drop is a diagnostic. Two shuffles are
not a confidence interval. Context-only columns are constant within a query and can have little
direct ranking effect while interacting with item features.

In [ ]:
permutations = pd.DataFrame([{'family': family, 'mean': row['mean_weighted_recall_drop'],
    'minimum': min(row['repeat_drops']), 'maximum': max(row['repeat_drops'])}
    for family,row in interpretation['group_permutation'].items()]).sort_values('mean')
fig = go.Figure(go.Bar(x=permutations['mean']*100, y=permutations['family'], orientation='h',
    marker_color=COLORS['core'], error_x=dict(type='data', symmetric=False,
        array=(permutations['maximum']-permutations['mean'])*100,
        arrayminus=(permutations['mean']-permutations['minimum'])*100)))
fig.update_xaxes(title='Weighted Recall@20 drop (percentage points)')
show(fig, 'Repeat and intent evidence matter · whiskers span two shuffles', 440)
print(f"Diagnostic sample: {interpretation['selection_sessions']:,} sessions, "
      f"{interpretation['candidate_rows']:,} candidate rows; weighted recall {interpretation['sample_score']['weighted_recall_at_20']:.5f}.")

## Measure feature cost on identical work

The benchmark uses the same 32 selection prefixes and 400-candidate policy for every feature
set. The first pass warms the engine; two measured passes provide 64 query timings. It includes
candidate generation and requested feature construction in one process. It excludes model
prediction, storage/network overhead, and online serving. The hardware is a SageMaker
**ml.c7i.16xlarge (64 logical CPUs, approximately 124 GiB available RAM)**.

In [ ]:
timing = pd.DataFrame(interpretation['feature_benchmark']).T
labels = ['Broad catalog · 1,482','Shortlist · 128','Selected · 102']
fig = go.Figure()
for metric, color in [('p50_ms', COLORS['core']), ('p95_ms', COLORS['selected'])]:
    values = timing.loc[['broad_catalog','screened','selected_models'], metric]
    fig.add_bar(x=labels, y=values, name=metric.replace('_ms',''), marker_color=color,
        text=values.map(lambda x:f'{x:.2f} ms'), textposition='outside')
fig.update_layout(barmode='group')
fig.update_yaxes(title='Warm per-query feature time (ms)', range=[0,27])
show(fig, 'Computing selected features avoids most catalog-wide overhead', 440)
print(interpretation['benchmark_scope'])

## Inspect failures and set the limits of the claim

Short prefixes dominate the query count. The overall weighted metric pools task denominators;
it is not an unweighted average of the slice scores. In the 2–5-event slice the selected model
slightly loses to fusion, despite improving the overall result. Longer-session orders explain
much of the advantage. This is diagnostic analysis after the model was frozen, not another
model-selection pass.

In [ ]:
slices = []
for label in ('1','2-5','6-20','21+'):
    row = evaluation['prefix_slices'][label]
    slices.append({'Observed events':label, 'Sessions':row['sessions'],
        **{name:row[name]['weighted_recall_at_20'] for name in ('fusion','core','selected')}})
display(pd.DataFrame(slices).style.format({'Sessions':'{:,}', 'fusion':'{:.5f}', 'core':'{:.5f}', 'selected':'{:.5f}'}).hide(axis='index'))
checks = pd.DataFrame([
    ['Target reconstruction', f"{audit['source']['reconstructed_labels']:,} labels; zero differences"],
    ['Metric coverage', f"{audit['statistics']['parts']} parts; {audit['statistics']['sessions']:,} sessions"],
    ['Model selection', f"{audit['ablations']['native_models']} native models; predeclared rule verified"],
    ['Prediction replay', f"{audit['native_replay']['comparisons']:,} independent checks; zero differences"],
    ['Raw conversion provenance', 'Original raw JSONL identity retained; conversion not independently replayed'],
], columns=['Audit', 'Evidence'])
display(checks.style.hide(axis='index'))

## What this establishes

The selected representation improves the compact ranking control in all nine audited runs across three temporal windows and three training seeds. The original reference result is 0.584392 weighted Recall@20. Matched gains range from +1.791 to +2.180 percentage points, and every paired 95% session-bootstrap interval is above zero.

All nine selection procedures choose the variant without direct source-score features for clicks, carts and orders. Retrieval still generates the candidates; graph and intent features retain indirect retrieval information. This repeated negative ablation is evidence for simplifying the ranker.

The study supports reproducible offline gains on this historical dataset. It does not establish online business lift, causal feature effects or state-of-the-art performance. Earlier neural benchmarks use different protocols. [Notebook 10](10_competition_inference.ipynb) demonstrates official-prefix inference with the original frozen reference weights and records the Kaggle submission state.


## Completed robustness extension

The frozen protocol repeated the complete procedure across **three windows × three model seeds**. Earlier windows built retrieval from permitted history and screened features using only their fitting cohorts. Seeds within a window share the prepared inputs; only the model seed changes.

**All nine cells are independently audited.** The table includes every planned outcome. These are saved verified results, not a live cloud dashboard.


In [ ]:
import tomllib

robustness = json.loads((ROOT / 'reports/robustness/progress.json').read_text())
robustness_plan = tomllib.loads((ROOT / 'configs/robustness.toml').read_text())
expected_plan = hashlib.sha256(json.dumps(robustness_plan, sort_keys=True, separators=(',', ':'), allow_nan=False).encode()).hexdigest()
assert robustness['protocol_id'] == expected_plan, 'Stale robustness protocol'
expected_cells = {f"{w['name']}_seed_{s}" for w in robustness_plan['windows'] for s in robustness_plan['model_seeds']}
assert {r['cell_id'] for r in robustness['cells']} == expected_cells
reference_cell = next(r for r in robustness['cells'] if r['cell_id'] == 'reference_seed_20260908')
assert reference_cell['weighted_recall_at_20'] == evaluation['scores']['selected']['weighted_recall_at_20']
assert all(r['weighted_recall_at_20'] is None for r in robustness['cells'] if r['status'] in {'planned', 'running'})
progress_table = pd.DataFrame(robustness['cells'])[['window', 'model_seed', 'status', 'weighted_recall_at_20']]
progress_table.columns = ['Temporal window', 'Model seed', 'Observed state', 'Weighted Recall@20']
progress_table['Observed state'] = progress_table['Observed state'].replace({'verified_reference': 'Verified reference', 'verified': 'Verified replication', 'running': 'Running; score pending', 'planned': 'Planned'})
display(progress_table.style.format({'Model seed': '{:.0f}', 'Weighted Recall@20': '{:.6f}'}, na_rep='Pending').hide(axis='index'))
print(f"Status observed at {robustness['observed_at_utc']}. This is saved execution evidence, not a live AWS dashboard.")
print(f"Verified full submission files: {robustness['verified_full_submissions']}; valid Kaggle scores recorded: {robustness['valid_kaggle_scored_submissions']}.")


## Does the feature gain repeat across time and training seeds?

**Yes, within this frozen study.** All nine selected rankers beat their matched compact controls. Gains range from **+1.791 to +2.180 percentage points**. The all-seed score ranges are **0.566465–0.566897** for the early window, **0.590465–0.590759** for the middle window, and **0.584392–0.584988** for the reference window.

The highest absolute offline score is **0.590759**, middle window seed **20260908**. Within the reference cohort, seed **20260910** is highest at **0.584988**. Different windows contain different sessions, so these maxima are descriptive; they do not identify a universally better model. The original reference seed **20260908** remains frozen for submission with its **0.584392** evaluation. No seed is selected using reserved evaluation outcomes.

The error bars are paired 95% session-bootstrap intervals conditional on each fitted model pair. Seeds within a window share their evaluation sessions and must not be pooled as independent observations. Earlier histories overlap and the checks are retrospective.

The original events, every reserved-session metric, 24 native models and eight ablations per replication, sampled model replay, and both 1,000-resample bootstrap comparisons were independently checked. The raw JSONL conversion was not replayed. Full model, per-objective and feature-selection details remain in the tables below.


In [ ]:
# Plot the verified comparison only; no synthetic or pending scores enter the chart.
replications = json.loads((ROOT / 'reports/robustness/comparison.json').read_text())
assert replications['protocol_id'] == expected_plan
for name, expected in replications['source_files'].items():
    source = (ROOT / name).resolve()
    assert source.is_relative_to(ROOT.resolve())
    assert hashlib.sha256(source.read_bytes()).hexdigest() == expected, name
verified_ids = {r['cell_id'] for r in robustness['cells'] if r['status'].startswith('verified')}
assert {r['cell_id'] for r in replications['rows']} == verified_ids
rows = replications['rows']
labels = [f"{r['window'].title()} · {r['model_seed']}" for r in rows]
fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.13,
    subplot_titles=('Recommendation quality', 'Gain over compact control'))
for key, label in [('fusion', 'Candidate fusion'), ('core', 'Compact control'), ('selected', 'Selected representation')]:
    fig.add_trace(go.Bar(name=label, orientation='h', y=labels,
        x=[r['scores'][key]['weighted_recall_at_20'] for r in rows], marker_color=COLORS[key],
        text=[f"{r['scores'][key]['weighted_recall_at_20']:.4f}" for r in rows], textposition='outside', cliponaxis=False,
        hovertemplate='%{y}<br>Weighted Recall@20: %{x:.6f}<extra>%{fullData.name}</extra>'), row=1, col=1)
gains = [100*r['paired_intervals']['core']['absolute_gain'] for r in rows]
limits = [[100*v for v in r['paired_intervals']['core']['gain_interval']] for r in rows]
fig.add_trace(go.Scatter(y=labels, x=gains, mode='markers+text', showlegend=False,
    text=[f'{g:+.3f} pp' for g in gains], textposition='top center', textfont=dict(size=12),
    marker=dict(size=13, color=COLORS['selected']),
    error_x=dict(type='data', symmetric=False,
        array=[hi-g for g, (lo, hi) in zip(gains, limits)],
        arrayminus=[g-lo for g, (lo, hi) in zip(gains, limits)], thickness=2, width=7),
    hovertemplate='%{y}<br>Selected minus compact: %{x:.3f} pp<extra>Paired 95% interval</extra>'), row=1, col=2)
fig.add_vline(x=0, line_dash='dot', line_color='#64748B', row=1, col=2)
score_limit = max(0.75, min(1.0, 1.1*max(r['scores'][key]['weighted_recall_at_20'] for r in rows for key in ('fusion', 'core', 'selected'))))
fig.update_xaxes(title_text='Weighted Recall@20', range=[0, score_limit], row=1, col=1)
fig.update_xaxes(title_text='Percentage points', rangemode='tozero', row=1, col=2)
fig.update_yaxes(type='category', categoryorder='array', categoryarray=labels,
    autorange='reversed')
height = max(560, 200 + 78*len(labels))
window_label = 'window represented' if replications['verified_windows'] == 1 else 'windows represented'
fig.update_layout(template='plotly_white', width=1100, height=height, barmode='group',
    title=dict(text=f"<b>Feature gains across time and training seeds</b><br><sup>{replications['verified_cells']} of {replications['planned_cells']} planned cells verified · {replications['verified_windows']} {window_label}</sup>",
               x=0.03, y=1-44/height, yanchor='top'),
    font=dict(family='Arial', size=14, color='#1E293B'),
    margin=dict(l=190, r=45, t=135, b=100),
    legend=dict(orientation='h', x=0, y=-60/(height-235)), paper_bgcolor='white', plot_bgcolor='white')
fig.show()
comparison_rows = []
for r in rows:
    ci = r['paired_intervals']['core']
    comparison_rows.append({'Window': r['window'], 'Seed': r['model_seed'], 'Sessions': r['sessions'],
        **{name: r['scores'][name]['weighted_recall_at_20'] for name in ('fusion', 'core', 'selected')},
        'Gain (pp)': 100*ci['absolute_gain'], '95% lower (pp)': 100*ci['gain_interval'][0],
        '95% upper (pp)': 100*ci['gain_interval'][1]})
display(pd.DataFrame(comparison_rows).style.format({**{name: '{:.6f}' for name in ('fusion', 'core', 'selected')},
    **{name: '{:+.3f}' for name in ('Gain (pp)', '95% lower (pp)', '95% upper (pp)')},
    'Sessions': '{:,.0f}'}).hide(axis='index'))
objective_rows = []
for r in rows:
    for objective in ('clicks', 'carts', 'orders'):
        objective_rows.append({'Window': r['window'], 'Seed': r['model_seed'], 'Objective': objective,
            **{name: r['scores'][name]['objectives'][objective]['recall_at_20'] for name in ('fusion', 'core', 'selected')},
            'Chosen variant': r['chosen'][objective], 'Features': r['selected_features'][objective]})
display(pd.DataFrame(objective_rows).style.format({name: '{:.6f}' for name in ('fusion', 'core', 'selected')}).hide(axis='index'))
for r in rows:
    probe = r['prediction_comparison']
    if probe is not None:
        changed = {o: v['ordered_top20_changed'] for o, v in probe['objectives'].items()}
        print(f"{r['window'].title()}, seed {r['model_seed']}: ordered top-20 predictions changed in {changed}, out of {probe['sessions']} sampled sessions per objective.")
print('Prediction probes compare models on the same temporal cohort; no later-window model is applied to the earlier evaluation.')
for window in robustness_plan['windows']:
    subset = [r for r in rows if r['window'] == window['name']]
    if len(subset) < 2:
        if subset:
            print(f"{window['name'].title()}: one verified seed; a seed range is not yet available.")
        continue
    values = [r['scores']['selected']['weighted_recall_at_20'] for r in subset]
    window_gains = [100*r['paired_intervals']['core']['absolute_gain'] for r in subset]
    print(f"{window['name'].title()} ({len(subset)} verified seeds): selected-score spread {100*(max(values)-min(values)):.3f} pp; gain range {min(window_gains):+.3f} to {max(window_gains):+.3f} pp.")
print('Ranges describe seeds within a window. They are not confidence intervals, and differences across windows are not training-seed variation.')


## Follow-up: learned similarities did not justify promotion

Two cutoff-safe Item2Vec models added action- and recency-conditioned similarities to the same candidate pools and rankers. All-action features, intent features, and both together were tested against the matched 102-feature baseline. The best weighted selection result rose only from **0.599523 to 0.599636**; the paired descriptive interval includes zero. This experiment does not establish a meaningful gain, and no model was promoted to Kaggle.

These are **development selection scores**, separate from the temporal evaluation above and the accepted **0.56842 private Kaggle score**. Repeated selection comparisons are exploratory. The next experiment tests wider symmetric and forward co-visitation while keeping candidate count and ranker settings fixed; pending results are not plotted.


In [ ]:
representation_path = ROOT / 'reports/research/representation_results.json'
assert hashlib.sha256(representation_path.read_bytes()).hexdigest() == '7e5f5fc004a89e87d104506c94228b6d5c9a19de315747ffdb510c58ea0635ae'
representation = json.loads(representation_path.read_text())
assert representation['status'] == 'passed'
rows = representation['variants']
labels = {'baseline': 'Baseline', 'with_all': '+ All-action Item2Vec',
          'with_intent': '+ Intent Item2Vec', 'with_both': '+ Both'}
fig = make_subplots(rows=1, cols=2, subplot_titles=['Weighted selection Recall@20', 'Per-task selection Recall@20'])
for arm, label in labels.items():
    values = rows[arm]
    color = COLORS['core'] if arm == 'baseline' else COLORS['selected'] if arm == 'with_intent' else COLORS['fusion']
    fig.add_scatter(x=[values['weighted_recall_at_20']], y=[label], mode='markers+text',
        text=[f"{values['weighted_recall_at_20']:.6f}"], textposition='top center',
        marker=dict(size=11, color=color), name=label, showlegend=False, row=1, col=1)
    fig.add_scatter(x=['Clicks', 'Carts', 'Orders'],
        y=[values['objectives'][o]['recall_at_20'] for o in ('clicks', 'carts', 'orders')],
        mode='lines+markers', name=label, row=1, col=2)
fig.update_xaxes(range=[.594, .603], title_text='Recall@20 · zoomed scale', row=1, col=1)
fig.update_yaxes(title_text='Recall@20', row=1, col=2)
show(fig, 'Learned similarities: no supported weighted improvement', height=500, left=180)
display(pd.DataFrame([{'Arm': labels[k], 'Weighted Recall@20': v['weighted_recall_at_20']}
                     for k,v in rows.items()]).style.format({'Weighted Recall@20': '{:.6f}'}))


## Wider retrieval: coverage improved, ranking declined

Both wider graphs retrieved more relevant candidates, but their replacement pipelines scored below the unchanged baseline. These are development-selection measurements, not Kaggle scores. Neither arm is promoted. The subsequent fixed-candidate experiments below evaluate their graph affinities as additional features. Feature research remains open.


In [ ]:
retrieval_path = ROOT / 'reports/research/retrieval_results.json'
assert hashlib.sha256(retrieval_path.read_bytes()).hexdigest() == 'e543e63e452e301b5baa3086e0b65af0c2109369cff4a58c877fa84ffc8f227a'
retrieval = json.loads(retrieval_path.read_text())
assert retrieval['status'] == 'passed'
rows = retrieval['arms']
labels = {'baseline': 'Baseline', 'wide_symmetric': 'Wide symmetric', 'wide_forward': 'Wide forward'}
fig = make_subplots(rows=1, cols=2, subplot_titles=['Final ranking', 'Candidate coverage ceiling'])
for arm, label in labels.items():
    item = rows[arm]
    for col, metric in ((1, 'weighted_recall_at_20'), (2, 'candidate_ceiling')):
        value = item[metric]
        if isinstance(value, dict):
            value = value['weighted_recall_at_20']
        fig.add_scatter(x=[value], y=[label], mode='markers+text',
            text=[f'{value:.6f}'], textposition='top center', marker=dict(size=12),
            name=label, legendgroup=arm, showlegend=False, row=1, col=col)
fig.update_xaxes(title_text='Weighted selection Recall@20', range=[.59, .605], row=1, col=1)
fig.update_xaxes(title_text='Weighted ceiling', range=[.69, .73], row=1, col=2)
show(fig, 'More candidate coverage did not produce better ranking', height=400, left=150)


## Feature research remains open

The original feature study and nine temporal replications establish their reported gains. They do not exhaust distinct high-value feature hypotheses. The versioned inventory below records tested original scopes, open extensions and exclusions caused by missing data. These counts are not a completion percentage. See [the coverage assessment](../docs/FEATURE_RESEARCH.md) for definitions, sources and leakage controls. New-feature promotion also requires preregistered temporal confirmation.


In [ ]:
coverage_inventory = json.loads((ROOT / 'configs/feature_research.json').read_text())
# Notebook execution copies published reports/configs; source/test hashes are checked by project_status.py.
for reference in coverage_inventory['evidence'].values():
    if Path(reference['path']).parts[0] in ('reports', 'configs'):
        assert hashlib.sha256((ROOT / reference['path']).read_bytes()).hexdigest() == reference['sha256']
coverage_rows = coverage_inventory['families']
open_rows = [row for row in coverage_rows if row['disposition'] == 'open']
print(f"Feature gate: {'OPEN' if open_rows else 'awaiting confirmation review'}; temporal confirmation: {coverage_inventory['temporal_confirmation']['status']}.")
display(pd.DataFrame([{'Area': row['name'], 'Evidence scope': row['scope'],
                       'Disposition': row['disposition']} for row in coverage_rows]))
print(coverage_inventory['next_task'])


## Complementary graph features: completed negative result

This feature-only experiment kept every baseline candidate, negative sample and original feature. Fitting-only redundancy screening retained 120 of 144 proposed columns. Symmetric and forward arms use 162 columns each; their union uses 222. All three variants scored below the 102-column baseline on the same 20,000 complete selection sessions. None is promoted.

The separate aggregate audit verifies all 12 native model files, feature schemas and tree counts; it recomputes pooled Recall@20, paired differences and per-task intervals. Candidate coverage matches for every session and objective. It does not independently replay all feature values or predictions. Intervals are exploratory, with no correction for repeated selection or multiple comparisons. Graph utility pruning, normalization and the other unresolved feature families remain open.


In [ ]:
graph_result = load('graph_feature_results.json')
graph_audit = load('graph_feature_audit.json')
graph_run = load('graph_feature_run.json')
assert graph_result['status'] == graph_audit['status'] == 'passed'
assert graph_result['study_id'] == graph_audit['study_id'] == graph_run['study_id']
assert hashlib.sha256((DATA / 'graph_feature_results.json').read_bytes()).hexdigest() == graph_run['results_sha256']
assert hashlib.sha256((DATA / 'graph_feature_audit.json').read_bytes()).hexdigest() == graph_run['audit_sha256']
assert graph_audit['models_verified'] == 12 and graph_audit['sessions_per_arm'] == 20000
assert graph_audit['candidate_coverage_equal_per_session_and_objective'] is True
assert not graph_audit['cross_family_screening_findings']
labels = {'baseline': 'Baseline', 'symmetric': '+ Symmetric affinities',
          'forward': '+ Forward affinities', 'both': '+ Both graph families'}
graph_colors = {'baseline': '#64748B', 'symmetric': '#0F766E', 'forward': '#2563EB', 'both': '#D97706'}
reference = graph_result['arms']['baseline']['weighted_recall_at_20']
rows = []
fig = make_subplots(rows=1, cols=2, subplot_titles=['Weighted selection Recall@20', 'Per-action change versus baseline'])
for arm, label in labels.items():
    value = graph_result['arms'][arm]
    score = value['weighted_recall_at_20']
    rows.append({'Arm': label, 'Features': len(value['features']), 'Weighted Recall@20': score,
                 'Change (pp)': 100 * (score - reference)})
    fig.add_scatter(x=[score], y=[label], mode='markers+text', text=[f'{score:.6f}'],
        textposition='top center', marker=dict(size=11, color=graph_colors[arm]), name=label, legendgroup=arm,
        showlegend=False, row=1, col=1)
    if arm != 'baseline':
        values = graph_audit['arms'][arm]['paired_gain']['objectives']
        points = [100 * values[o]['gain'] for o in ('clicks', 'carts', 'orders')]
        upper = [100 * (values[o]['ci95'][1] - values[o]['gain']) for o in ('clicks', 'carts', 'orders')]
        lower = [100 * (values[o]['gain'] - values[o]['ci95'][0]) for o in ('clicks', 'carts', 'orders')]
        fig.add_scatter(x=['Clicks', 'Carts', 'Orders'], y=points, mode='markers',
            marker=dict(size=8, color=graph_colors[arm]),
            error_y=dict(type='data', symmetric=False, array=upper, arrayminus=lower, color=graph_colors[arm]),
            name=label, legendgroup=arm, row=1, col=2)
fig.update_xaxes(title_text='Recall@20; zoomed scale', range=[.592, .603], row=1, col=1)
fig.update_yaxes(title_text='Percentage points; descriptive 95% intervals', row=1, col=2)
fig.add_hline(y=0, line_color='#64748B', line_dash='dot', row=1, col=2)
show(fig, 'Raw graph additions did not improve fixed-candidate ranking', height=520, left=190)
display(pd.DataFrame(rows).style.format({'Weighted Recall@20': '{:.6f}', 'Change (pp)': '{:+.4f}'}))
print('Managed job: Completed. Accepted Kaggle baseline: private 0.56842 / public 0.56862. Feature gate: OPEN.')


## Shopping progression and normalized graph evidence

Can recorded shopping-action order, gap-defined activity and popularity-adjusted graph evidence help on exactly the same candidates? The preregistered comparison adds 172 eligible formulas across four families: 40 funnel, 24 episode, 36 matched raw-graph controls and 72 normalized graph scores. Each arm is screened independently on fitting rows; all original 102 columns remain fixed. The saved baseline models are replayed without retraining, and must reproduce every original selection-session statistic.

Raw and normalized graph scores use the same historical graphs and the same last-item, recent-distinct and cart/order-distinct pools. Normalized scores use retained outgoing/incoming graph mass; they are not calibrated purchase probabilities. The sequence arm combines action transitions and gap-defined episodes, so it does not isolate either subfamily. [Definitions, domain evidence and preregistration](../docs/FEATURE_RESEARCH.md) describe the exact formulas and limitations.

The aggregate audit below verifies 15 native models across five arms and recomputes full-query metrics and paired whole-session intervals. It does not independently regenerate all raw features or challenger predictions. The previously used selection cohort and unadjusted 95% intervals support exploration; promotion requires a separately declared temporal confirmation.


In [ ]:
domain_result = load('domain_feature_results.json')
domain_audit = load('domain_feature_audit.json')
domain_run = load('domain_feature_run.json')
assert domain_result['status'] == domain_audit['status'] == domain_run['status'] == 'passed'
assert domain_result['study_id'] == domain_audit['study_id'] == domain_run['study_id']
assert domain_run['managed_status'] == 'Completed'
for name in ('results', 'audit', 'screening', 'slices'):
    assert hashlib.sha256((DATA / f'domain_feature_{name}.json').read_bytes()).hexdigest() == domain_run[f'{name}_sha256']
assert domain_audit['models_verified'] == 15 and domain_audit['sessions_per_arm'] == 20000
assert domain_audit['candidate_coverage_equal_per_session_and_objective'] is True
assert domain_result['arms']['baseline']['reused_reference'] is True
assert domain_run['baseline_models_reused'] == 3 and domain_run['new_models_fitted'] == 12
labels = {'baseline': 'Baseline replay', 'sequence': '+ Sequence / episodes',
          'raw_graph': '+ Raw graph pools', 'normalized_graph': '+ Normalized graph pools',
          'combined': '+ Sequence + normalized'}
colors = {'baseline': '#64748B', 'sequence': '#7C3AED', 'raw_graph': '#D97706',
          'normalized_graph': '#0F766E', 'combined': '#2563EB'}
reference = domain_result['arms']['baseline']['weighted_recall_at_20']
rows = []
fig = make_subplots(rows=1, cols=2, subplot_titles=['Weighted change versus baseline', 'Per-action change versus baseline'])
for arm_index, (arm, label) in enumerate(labels.items()):
    item = domain_result['arms'][arm]
    gain = 100 * (item['weighted_recall_at_20'] - reference)
    rows.append({'Arm': label, 'Features': len(item['features']),
                 'Weighted Recall@20': item['weighted_recall_at_20'], 'Change (pp)': gain,
                 **{o.title(): item['objectives'][o]['recall_at_20'] for o in ('clicks', 'carts', 'orders')}})
    if arm != 'baseline':
        comparison = domain_audit['arms'][arm]['paired_gain']
        interval = [100 * v for v in comparison['weighted_ci95']]
        fig.add_scatter(x=interval, y=[label, label], mode='lines',
            line=dict(color=colors[arm], width=3), showlegend=False, row=1, col=1)
        task_values = comparison['objectives']
        offset = (arm_index - 2.5) * .06
        positions = [j + offset for j in range(3)]
        for j, objective in enumerate(('clicks', 'carts', 'orders')):
            fig.add_scatter(x=[positions[j], positions[j]],
                y=[100 * v for v in task_values[objective]['ci95']], mode='lines',
                line=dict(color=colors[arm], width=2), showlegend=False, row=1, col=2)
        fig.add_scatter(x=positions, y=[100 * task_values[o]['gain'] for o in ('clicks', 'carts', 'orders')],
            mode='markers', marker=dict(size=9, color=colors[arm]), name=label,
            legendgroup=arm, row=1, col=2)
    fig.add_scatter(x=[gain], y=[label], mode='markers', marker=dict(size=11, color=colors[arm]),
        name=label, legendgroup=arm, showlegend=False, row=1, col=1)
fig.add_vline(x=0, line_color='#64748B', line_dash='dot', row=1, col=1)
fig.add_hline(y=0, line_color='#64748B', line_dash='dot', row=1, col=2)
fig.update_xaxes(title_text='Percentage points; descriptive 95% intervals', row=1, col=1)
fig.update_xaxes(tickvals=[0, 1, 2], ticktext=['Clicks', 'Carts', 'Orders'], row=1, col=2)
fig.update_yaxes(categoryorder='array', categoryarray=list(labels.values())[::-1], row=1, col=1)
fig.update_yaxes(title_text='Percentage points', row=1, col=2)
interval_values = [100 * value for arm in labels if arm != 'baseline'
                   for value in domain_audit['arms'][arm]['paired_gain']['weighted_ci95']]
padding = .06 * (max(interval_values) - min(interval_values))
fig.update_xaxes(range=[min(interval_values) - padding, max(interval_values) + padding], row=1, col=1)
show(fig, 'Domain hypotheses on unchanged candidate rows', height=650, left=210,
     margin=dict(l=210, r=40, t=85, b=150),
     legend=dict(orientation='h', y=-.22, x=0, yanchor='top',
                 entrywidth=.5, entrywidthmode='fraction'))
display(pd.DataFrame(rows).style.format({c: '{:.6f}' for c in ('Weighted Recall@20', 'Clicks', 'Carts', 'Orders')} | {'Change (pp)': '{:+.4f}'}))
raw_check = domain_audit['normalized_vs_raw']
print(f"Normalized versus matched raw pools: {100 * raw_check['weighted_gain']:+.4f} pp; descriptive 95% interval {[round(100 * v, 4) for v in raw_check['ci95']]} pp.")
print(f"Processing completed in {float(domain_run['processing_seconds']) / 60:.2f} minutes; estimated instance compute ${float(domain_run['estimated_instance_compute_usd']):.2f}, excluding storage, requests, logging and transfer.")
print('Accepted Kaggle baseline: private 0.56842 / public 0.56862. Feature engineering gate: OPEN.')


## Where can the new signals have support?

Nearly half of the selection sessions have one observed event. Only 2,997 of 20,000 have an observed cart/order, and only 1,308 have an observed gap over 30 minutes. These support counts explain where a feature can vary; they do not establish that it improves prediction.

The following predefined prefix-length slices retain complete query denominators. Their counts partition the cohort. Other recorded diagnostics cover repeated items, observed cart/order activity and long gaps. Those slice families overlap and describe query context, not target-item novelty or rarity. Differences below are descriptive point estimates, without slice-specific intervals or a promotion claim.


In [ ]:
domain_slices = load('domain_feature_slices.json')
domain_profile = load('domain_prefix_profile.json')
assert domain_slices['study_id'] == domain_run['study_id']
assert hashlib.sha256((DATA / 'domain_prefix_profile.json').read_bytes()).hexdigest() == domain_run['profile_sha256']
groups = {'prefix_1': '1 event', 'prefix_2_to_5': '2–5 events', 'prefix_6_to_20': '6–20 events', 'prefix_21_plus': '21+ events'}
reference_slices = domain_slices['arms']['baseline']
assert sum(reference_slices[g]['sessions'] for g in groups) == 20000
for arm, summary in domain_result['arms'].items():
    for objective in ('clicks', 'carts', 'orders'):
        for field in ('hits', 'denominator'):
            assert sum(domain_slices['arms'][arm][g]['objectives'][objective][field] for g in groups) == summary['objectives'][objective][field]
challengers = [arm for arm in labels if arm != 'baseline']
changes = np.array([[100 * (domain_slices['arms'][arm][g]['weighted_recall_at_20'] - reference_slices[g]['weighted_recall_at_20'])
                     for g in groups] for arm in challengers])
limit = max(.01, float(np.abs(changes).max()))
fig = go.Figure(go.Heatmap(z=changes,
    x=[f"{label}<br>N={reference_slices[g]['sessions']:,}" for g, label in groups.items()],
    y=[labels[arm] for arm in challengers], zmin=-limit, zmax=limit, zmid=0,
    colorscale='RdBu', reversescale=False, text=changes, texttemplate='%{text:+.3f}',
    colorbar=dict(title='Change (pp)')))
show(fig, 'Descriptive prefix-length differences', height=420, left=220)
display(pd.DataFrame([{'Observed property': name, 'Fitting sessions': domain_profile['roles']['fit']['slice_sessions'][name],
    'Selection sessions': domain_profile['roles']['selection']['slice_sessions'][name]} for name in ('prefix_1', 'has_repeat_item', 'observed_cart_or_order', 'gap_over_30_minutes')]))


## A task-specific follow-up hypothesis

The sequence arm has the highest click/cart point estimates; normalized graph features have the highest order point estimate. Recombining those saved models yields 0.601446 on this selection cohort. This is a **post-hoc diagnostic selected after outcomes were inspected**, not a sixth preregistered arm or an independently measured improvement. No additional fitting was needed. The selection rule prefers the baseline on exact ties, then the alphabetical arm name.

This optimistic score motivates fitting-only per-action feature screening and a frozen temporal confirmation; it cannot justify promotion. [Cawley and Talbot (2010)](https://www.jmlr.org/papers/v11/cawley10a.html) explain the evaluation bias induced by model selection. No interval is presented as independent evidence for this selected mixture.


In [ ]:
mix = domain_slices['same_cohort_task_mix']
assert mix['status'] == 'post_hoc_diagnostic_not_an_independent_experiment'
weights = {'clicks': .1, 'carts': .3, 'orders': .6}
mixture_rows = []
reconstructed = 0.
for objective, weight in weights.items():
    arm = mix['selected_arms'][objective]
    recall = domain_result['arms'][arm]['objectives'][objective]['recall_at_20']
    reconstructed += weight * recall
    mixture_rows.append({'Action': objective.title(), 'Chosen existing arm': labels[arm],
                         'Selection Recall@20': recall,
                         'Change versus baseline (pp)': 100 * (recall - domain_result['arms']['baseline']['objectives'][objective]['recall_at_20'])})
assert abs(reconstructed - mix['weighted_recall_at_20']) < 1e-12
assert abs(reconstructed - reference - mix['absolute_gain']) < 1e-12
display(pd.DataFrame(mixture_rows).style.format({'Selection Recall@20': '{:.6f}', 'Change versus baseline (pp)': '{:+.4f}'}))
print(f"Post-hoc selection score {reconstructed:.6f}; change {100 * mix['absolute_gain']:+.4f} pp. This is not an independently confirmed gain.")


## What this study resolves, and what remains open

All four weighted comparisons have descriptive 95% difference intervals spanning zero. The normalized arm's small weighted advantage combines better order recall with worse click recall; the combined arm does not remove that trade-off. Raw graph pools reduce the point estimate. The exact negative and uncertain results remain in the committed reports.

The sequence arm improves the one-event slice even though no observed action transition exists there. Static last-state/context features and changed model splits can still affect its predictions; the slice does not identify a causal transition benefit. Long-prefix comparisons are small (409 sessions with 21+ observed events). The normalized and combined cart models, and the combined click model, select the first boosting iteration, which warrants per-task utility and stopping-curve diagnosis rather than a stability claim.

This motivated the follow-up below: reuse the certified feature caches to separate funnel from episode and row from degree normalization, and screen utility on fitting-only session groups for each action. Freeze the selected schemas and routing before separately declared temporal confirmation. One seed, one reused development cohort and redundancy screening do not exhaust these hypotheses. The broader inventory still has 14 open families; feature engineering remains **OPEN**. The accepted Kaggle baseline stays 0.56842 private / 0.56862 public.


## A bounded feature-screening follow-up

The follow-up freezes the original 102 columns and compares no additions, a shared 32-column shortlist, and separate 32-column shortlists. Screening uses only 5,120 fitting sessions, with support across three session-grouped fitting folds. All three ranking arms use the same 2,048 selection sessions and all 400 candidates per query. The final temporal evaluation is not accessed.

This is a small systematic sample of reused development cohorts. It tests whether supervised pruning is promising at fixed model/data budgets. It does not establish a leaderboard score or replace temporal confirmation. [Protocol, domain rationale and limits](../docs/TASK_FEATURE_PILOT.md).

In [ ]:
task_run = load('task_feature_run.json')
for name, expected in task_run['files'].items():
    assert hashlib.sha256((DATA / name).read_bytes()).hexdigest() == expected, name
task_result = load('task_feature_results.json')
task_audit = load('task_feature_audit.json')
task_screen = load('task_feature_screening.json')
assert task_run['observed']['ProcessingJobStatus'] == 'Completed'
assert task_audit['status'] == task_result['status'] == 'passed'
assert task_audit['native_rankers_verified'] == 9
assert task_audit['training_free_replay']['training_calls'] == 0
assert task_audit['training_free_replay']['all_models_and_metrics_identical']
assert task_result['evaluation_access'] is False and task_result['kaggle_promotion'] is False
assert task_result['inputs']['fit']['sessions'] == 5120
assert task_result['inputs']['selection']['sessions'] == 2048
assert task_screen['study_id'] == task_result['study_id'] == task_run['study_id']
arm_labels = {'baseline': '102-column baseline', 'shared': 'Shared 32 additions', 'per_task': 'Per-task 32 additions'}
small_reference = task_result['arms']['baseline']['weighted_recall_at_20']
rows = []
for arm, label in arm_labels.items():
    summary = task_result['arms'][arm]
    pooled = sum(weights[action] * summary['objectives'][action]['hits'] / summary['objectives'][action]['denominator'] for action in weights)
    assert abs(pooled - summary['weighted_recall_at_20']) < 1e-12
    rows.append({'Arm': label, 'Weighted Recall@20': pooled, 'Change vs baseline (pp)': 100 * (pooled - small_reference),
                 **{action.title(): summary['objectives'][action]['recall_at_20'] for action in weights}})
display(pd.DataFrame(rows).style.format({name: '{:.6f}' for name in ['Weighted Recall@20', 'Clicks', 'Carts', 'Orders']} | {'Change vs baseline (pp)': '{:+.3f}'}))
print(f"Managed job completed in {task_run['processing_seconds'] / 60:.2f} processing minutes; estimated instance compute ${task_run['estimated_instance_compute_usd']:.3f}.")
print('All nine native rankers verified; restart made zero training calls and reproduced every result.')


In [ ]:
fig = go.Figure()
for arm, color in [('shared', '#0F766E'), ('per_task', '#2563EB')]:
    deltas = [100 * (task_result['arms'][arm]['objectives'][action]['recall_at_20'] - task_result['arms']['baseline']['objectives'][action]['recall_at_20']) for action in weights]
    fig.add_trace(go.Bar(name=arm_labels[arm], x=[action.title() for action in weights], y=deltas,
        marker_color=color, text=deltas, texttemplate='%{text:+.3f}', textposition='outside'))
fig.add_hline(y=0, line_color='#64748B', line_width=1)
show(fig, 'Small development pilot: orders drive the shared-shortlist gain', height=440,
     yaxis_title='Change vs matched baseline (percentage points)', barmode='group')
interval = task_result['arms']['shared']['versus_baseline']['gain_interval']
difference = task_result['arms']['per_task']['versus_shared']
print(f"Shared minus baseline: {100 * (task_result['arms']['shared']['weighted_recall_at_20'] - small_reference):+.3f} pp; descriptive paired 95% interval [{100 * interval[0]:+.3f}, {100 * interval[1]:+.3f}] pp.")
print(f"Per-task minus shared: {100 * difference['absolute_gain']:+.3f} pp; descriptive paired 95% interval [{100 * difference['gain_interval'][0]:+.3f}, {100 * difference['gain_interval'][1]:+.3f}] pp.")
print('Intervals condition on these models and reused systematic development samples; they do not account for training variability or repeated research selection.')


## Decision: validate the shared shortlist; do not scale the per-task rule

The shared arm improves weighted recall by **2.599 percentage points**, mostly through orders, while losing three click hits. The per-task arm improves on the baseline but fails to beat the shared control. The preregistered condition for scaling that exact per-task procedure is therefore not met.

The shared 32 additions comprise 11 funnel/state features, six episode features and 15 graph affinities. Their individual value remains unresolved: fitting-fold gain support is not an ablation. The next bounded stage uses the [frozen shared schema](../configs/shared_feature_validation.json) with 12,800 fitting and 5,120 development selection sessions. If the larger matched gain warrants more compute, ablate funnel, episode, raw, row-normalized and degree-normalized graph blocks, then preregister temporal/seed confirmation.

**0.604069 is not a Kaggle score.** The accepted score remains **0.56842 private / 0.56862 public**, versus the historical winning private score **0.60503**. No challenger is promoted. All 14 unresolved feature families remain open, including demand drift, smoothed propensities, latent/session-neighbor representations, multiple intents and retrieval complementarity. These possibilities require distinct hypotheses and tests; this pilot does not exhaust feature engineering.
